# Module 7: Solutions Notebook

Reference solutions for the AI in Banking and Finance exercises.
Use this to verify your work after completing the participant notebook.

**Do not open this notebook until you have attempted all exercises.**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

use_cases = pd.read_csv("../data/ai_use_cases_banking.csv")
prompts = pd.read_csv("../data/prompt_templates.csv")

In [ ]:
# --- Solution: Exercise 1 - Use Case Landscape ---

# A: Department distribution
dept_counts = use_cases["department"].value_counts()
print("Department distribution:")
print(dept_counts)
print()

# Key finding: Operations and Customer Service have the most use cases,
# reflecting the high volume of repetitive tasks suitable for AI.
# Compliance also features heavily due to regulatory burden.

# B: AI type comparison
ai_type_summary = use_cases.groupby("ai_type").agg(
    count=("use_case_id", "count"),
    avg_value=("estimated_value_sar", "mean"),
    high_risk_pct=("risk_rating", lambda x: (x == "High").mean() * 100)
).round(1)
print("GenAI vs Traditional ML comparison:")
print(ai_type_summary)
print()

# Key finding: Traditional ML use cases have higher average value and
# a higher proportion of high-risk ratings. This reflects the fact that
# Traditional ML is used for established, high-stakes processes (credit
# scoring, fraud detection) while GenAI is applied more to support and
# augmentation tasks.

# C: Cross-tabulation
cross = pd.crosstab(use_cases["department"], use_cases["ai_type"], margins=True)
print("Department x AI Type (with totals):")
print(cross)

In [ ]:
# --- Solution: Exercise 2 - Risk and Compliance ---

# A: Risk-compliance matrix
matrix = pd.crosstab(
    use_cases["risk_rating"],
    use_cases["compliance_impact"],
    margins=True
)
print("Risk x Compliance matrix:")
print(matrix)
print()

# B: High-risk, high-compliance use cases
critical = use_cases[
    (use_cases["risk_rating"] == "High") &
    (use_cases["compliance_impact"] == "High")
].copy()

print(f"Critical oversight use cases ({len(critical)}):")
for _, row in critical.iterrows():
    print(f"  {row['use_case_id']}: {row['use_case_name']} ({row['department']}) - {row['current_status']}")
print()

# These use cases require: mandatory human review, audit trails,
# model governance documentation, and regulatory approval before
# production deployment.

# C: Value by risk
value_analysis = use_cases.groupby("risk_rating").agg(
    total_value=("estimated_value_sar", "sum"),
    mean_value=("estimated_value_sar", "mean"),
    count=("use_case_id", "count")
).round(0)
print("Value by risk rating:")
print(value_analysis)
print()

# Key insight: High-risk use cases represent the majority of total
# value. This creates a tension between value realisation and
# governance requirements that leadership must address.

# D: Stretch - high-risk adoption readiness
print("High-risk use case readiness:")
status_dist = critical["current_status"].value_counts()
for status, count in status_dist.items():
    pct = count / len(critical) * 100
    print(f"  {status}: {count} ({pct:.0f}%)")
print()
print("Interpretation: A significant share of high-risk use cases")
print("are still in evaluation, indicating appropriate caution.")
print("Those in production have passed through governance review.")

In [ ]:
# --- Solution: Exercise 3 - Prompt Template Analysis ---

prompts["constraint_length"] = prompts["constraints"].str.len()
prompts["constraint_count"] = prompts["constraints"].str.count(";") + 1

print("Constraint analysis by risk level:")
constraint_analysis = prompts.groupby("risk_level").agg(
    avg_constraints=("constraint_count", "mean"),
    avg_char_length=("constraint_length", "mean"),
    templates=("template_id", "count")
).round(1)
print(constraint_analysis)
print()

# Key finding: Higher-risk prompts tend to have more constraints
# and longer constraint text. This is appropriate: more sensitive
# outputs require more guardrails in the prompt itself.

# Stretch: Example improved high-risk prompt
print("Example improved prompt for Credit Memo Drafting (PT-005):")
print()
print("Original constraints:")
print("  Follow internal memo format; include all required sections; mark assumptions clearly")
print()
print("Improved constraints:")
print("  Follow internal credit memo format v3.2;")
print("  include all 7 required sections per AJB Credit Policy CP-2024-01;")
print("  mark every assumption with [ASSUMPTION] tag;")
print("  do not state credit decisions or approvals;")
print("  include disclaimer: 'Draft for review only, not a credit decision';")
print("  flag any data older than 90 days as potentially stale;")
print("  output must not exceed 2 pages")
print()
print("Rationale: Added policy references, explicit prohibitions,")
print("staleness checks, and output length limits. Each constraint")
print("reduces a specific failure mode.")

In [ ]:
# --- Solution: Exercise 4 - Prioritisation Model ---

risk_score = {"Low": 3, "Medium": 2, "High": 1}
complexity_score = {"Low": 3, "Medium": 2, "High": 1}
status_score = {"Production": 4, "Pilot": 3, "Evaluation": 2}

scored = use_cases.copy()
scored["risk_pts"] = scored["risk_rating"].map(risk_score)
scored["complexity_pts"] = scored["implementation_complexity"].map(complexity_score)
scored["status_pts"] = scored["current_status"].map(status_score)
scored["value_pts"] = (scored["estimated_value_sar"] / scored["estimated_value_sar"].max() * 3).round(1)
scored["priority"] = (scored["risk_pts"] + scored["complexity_pts"] + scored["status_pts"] + scored["value_pts"]).round(1)

ranked = scored.sort_values("priority", ascending=False)
print("Full priority ranking:")
print(ranked[["use_case_id", "use_case_name", "department", "priority"]].to_string(index=False))
print()

# Interpretation of top 5:
print("Top 5 analysis:")
for i, (_, row) in enumerate(ranked.head(5).iterrows(), 1):
    print(f"  {i}. {row['use_case_name']} (Score: {row['priority']})")
    print(f"     Department: {row['department']}, Risk: {row['risk_rating']}, Status: {row['current_status']}")
print()

# The top-scoring use cases tend to be low-risk, already in production,
# with reasonable value. This is expected: the scoring model favours
# "safe bets" that are easy to govern.
#
# For a more innovation-focused bank, you would increase the weight
# of value_pts and decrease risk_pts weight. For a more conservative
# bank, you would increase risk_pts and add a compliance_pts dimension.
#
# Missing data that would improve the model:
# - Time to implement (months)
# - Required team skills and availability
# - Vendor dependency
# - Data readiness score
# - Regulatory pre-approval status
# - Strategic alignment score from leadership

print("Model limitations:")
print("  - Equal weights may not reflect AJB strategic priorities")
print("  - Missing: data readiness, team capacity, vendor risk")
print("  - Status score rewards incumbency over innovation")
print("  - No time-to-value dimension")